# 🔎 AIOS Embeddings & RAG Index Build

Генерация эмбеддингов для всей базы знаний AIOS на **GPU Colab** (BAAI/bge-m3, nomic-embed-text) и сохранение векторного индекса в **ChromaDB** для мгновенного поиска на VPS.

**T4 GPU / High-RAM CPU**.

1. Загрузите `corpus.jsonl` (с VPS: папка `data/rag/`, получить `python aios_core/rag/index_builder.py --export`).
2. Выполните ячейки.
3. Скачайте папку `chroma_colab/` на VPS и запустите `scripts/import_colab_index.py --src <архив> --extract`.

In [ ]:
!pip install -q chromadb sentence-transformers torch
from sentence_transformers import SentenceTransformer
import torch
print('✅ Установлено, CUDA:', torch.cuda.is_available())

In [ ]:
# === ЯЧЕЙКА 2: Загрузка корпуса ===
import json, os
chunks = [json.loads(l) for l in open('corpus.jsonl', encoding='utf-8') if l.strip()]
print('✅ Чанков в корпусе:', len(chunks))
print('Пример:', chunks[0]['text'][:120])

In [ ]:
# === ЯЧЕЙКА 3: Модель эмбеддингов ===
# BAAI/bge-m3 (мультиязычная, 1024) — рекомендована. Альтернатива: nomic-embed-text-v1.5
model = SentenceTransformer('BAAI/bge-m3')
print('✅ Модель загружена, размерность:', model.get_sentence_embedding_dimension())

In [ ]:
# === ЯЧЕЙКА 4: Генерация эмбеддингов (батчами) ===
texts = [c['text'] for c in chunks]
embs = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
print('✅ Эмбеддинги:', embs.shape)

In [ ]:
# === ЯЧЕЙКА 5: Сохранение в ChromaDB ===
import chromadb
from chromadb.config import Settings
client = chromadb.PersistentClient(path='chroma_colab', settings=Settings(anonymized_telemetry=False))
col = client.get_or_create_collection('aios_knowledge')
# добавляем батчами (экономно по памяти)
B = 512
for i in range(0, len(chunks), B):
    c = chunks[i:i+B]
    col.add(ids=[x['id'] for x in c],
            documents=[x['text'] for x in c],
            metadatas=[x['metadata'] for x in c],
            embeddings=embs[i:i+B].tolist())
print('✅ В коллекции:', col.count())

In [ ]:
# === ЯЧЕЙКА 6: Тест поиска ===
res = col.query(query_texts=['как зарегистрировать сервис в реестре Colab?'], n_results=3)
for d in res['documents'][0]:
    print(' -', d[:120])
print('\n✅ Индекс готов')
!tar -czf chroma_colab.tar.gz chroma_colab
print('Скачайте chroma_colab.tar.gz на VPS и запустите: scripts/import_colab_index.py --src chroma_colab.tar.gz --extract')